# 04 · Validate — per-setting trade-offs + the Pareto map

**Standard slot:** *validate (in silico).* **For Project 02 this is the core science:** for each
setting, summarize recovery vs recapitulation vs diversity vs solubility, then find the
**Pareto-optimal** settings on the foldability ↔ diversity ↔ solubility frontier (D3 part 2). Also
compares consensus-design vs single-sequence `[extension]`.

Needs `results/sequences.csv` with the recapitulation columns from nb 03.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Per-setting metric table

Group the swept sequences by (temperature, noise, seqs) and compute, per cell: mean recovery,
**recapitulation rate** (fraction with scRMSD < 2 Å = foldability), mean per-position entropy
(diversity), and mean solubility proxies. This is the table the cheat-sheet is read off.

In [ ]:
import pandas as pd, numpy as np
from mpnn_tools import shannon_entropy, sequence_recovery

seqs = pd.read_csv("results/sequences.csv")
if "scrmsd" not in seqs:
    raise RuntimeError("Run notebook 03 first to add recapitulation columns.")

# Recovery needs a reference per backbone; in the dry run use the low-temp mock consensus as native.
from mpnn_tools import run_mpnn
ref_by_bb = {bb: run_mpnn(bb, temperature=0.01, noise=0.0, n_seqs=1, tool="mock")[0].sequence
             for bb in seqs["backbone"].unique()}
seqs["recovery"] = [sequence_recovery(ref_by_bb[bb], s)
                    for bb, s in zip(seqs["backbone"], seqs["sequence"])]

def per_position_entropy(group):
    # diversity is per-backbone (sequences for the same backbone are aligned), then averaged
    vals = [shannon_entropy(g["sequence"].tolist())["mean"]
            for _, g in group.groupby("backbone") if len(g) > 1]
    return float(np.mean(vals)) if vals else 0.0

agg = []
for (t, n, s), g in seqs.groupby(["temperature", "noise", "n_seqs"]):
    agg.append(dict(temperature=t, noise=n, n_seqs=s,
                    mean_recovery=g["recovery"].mean(),
                    recap_rate=(g["scrmsd"] < 2.0).mean(),         # foldability
                    mean_entropy=per_position_entropy(g),          # diversity
                    mean_net_charge=g["net_charge"].mean(),
                    mean_hp_frac=g["hydrophobic_fraction"].mean(),
                    mean_camsol_like=g["camsol_like"].mean()))     # solubility proxy
settings = pd.DataFrame(agg)
settings.to_csv("results/settings_summary.csv", index=False)
print("EXAMPLE_DATA (mock backend) — per-setting summary:")
settings.round(3)

## 2 · The Pareto-optimal settings

A setting is **Pareto-optimal** if no other setting beats it on *every* objective at once. We
maximize three objectives: foldability (`recap_rate`), diversity (`mean_entropy`), solubility
(`mean_camsol_like`). The frontier — not a single winner — is the deliverable.

In [ ]:
def pareto_front(df, objectives):
    """Return rows not dominated on all objectives (all maximized)."""
    M = df[objectives].values
    keep = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        for j in range(len(df)):
            if i != j and np.all(M[j] >= M[i]) and np.any(M[j] > M[i]):
                keep[i] = False
                break
    return df[keep]

OBJ = ["recap_rate", "mean_entropy", "mean_camsol_like"]
front = pareto_front(settings, OBJ).sort_values("recap_rate", ascending=False)
print(f"{len(front)}/{len(settings)} settings are Pareto-optimal on {OBJ}:")
front.round(3)

### Visualize the trade-off

Foldability vs diversity, colored by the solubility proxy. The Pareto front (ringed) is what the
cheat-sheet recommends from, by goal. (Mock data here is illustrative — label any such figure
`EXAMPLE_DATA`.)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4.5))
sc = ax.scatter(settings["recap_rate"], settings["mean_entropy"],
                c=settings["mean_camsol_like"], cmap="viridis", s=60, edgecolor="k", lw=0.3)
ax.scatter(front["recap_rate"], front["mean_entropy"], facecolors="none",
           edgecolors="red", s=180, lw=1.6, label="Pareto-optimal")
for _, row in settings.iterrows():
    ax.annotate(f"T{row.temperature}/N{row.noise}", (row.recap_rate, row.mean_entropy),
                fontsize=6, alpha=0.6)
ax.set_xlabel("recapitulation rate  (foldability)")
ax.set_ylabel("mean per-position entropy  (diversity)")
ax.set_title("MPNN settings trade-off  [EXAMPLE_DATA — mock backend]")
fig.colorbar(sc, label="camsol_like (solubility proxy, NOT real CamSol)")
ax.legend(); plt.tight_layout(); plt.savefig("results/pareto.png", dpi=150); plt.show()

## 3 · Consensus-design vs single-sequence `[extension]`

Does the per-position **consensus** of a backbone's N sequences recapitulate better than the single
best sequence? Consensus often improves stability/expression in the literature — test it here per
backbone and report the win rate (honestly, including ties/losses).

In [ ]:
from collections import Counter
import numpy as np

def consensus(seqs_list):
    L = min(len(s) for s in seqs_list)
    return "".join(Counter(s[i] for s in seqs_list).most_common(1)[0][0] for i in range(L))

# Compare at a single representative setting (e.g., temp 0.2, noise 0.1).
sub = seqs[(seqs.temperature == 0.2) & (seqs.noise == 0.1)]
import hashlib
def mock_recapitulate(s):
    h = int(hashlib.sha256(s.encode()).hexdigest(), 16); return 0.8 + (h % 350)/100.0
wins = 0; total = 0
for bb, g in sub.groupby("backbone"):
    sl = g["sequence"].tolist()
    if len(sl) < 2: continue
    cons_scrmsd = mock_recapitulate(consensus(sl))
    best_single = g["scrmsd"].min()
    wins += int(cons_scrmsd < best_single); total += 1
print(f"consensus beat best-single recapitulation in {wins}/{total} backbones "
      f"[EXAMPLE_DATA — mock]. Report this honestly, including losses, on real data.")

## 4 · ProteinMPNN vs ESM-IF (and FAMPNN if available) `[extension]`

Re-run the sweep (or the best settings) with **ESM-IF** (inverse folding) as a second designer and
compare recovery/recapitulation/diversity head-to-head. Scaffold below — wire in the real backends
on Colab; report per-tool distributions, not single bests.

In [ ]:
# Scaffold (implement on Colab with the real backends):
# for tool in ("proteinmpnn", "esm_if"):           # add "fampnn" if a public release exists
#     pool = sweep_with(tool, BACKBONES, best_settings)
#     recapitulate(pool); summarize(pool)
# compare recovery / recap_rate / entropy across tools (boxplots), report N per tool.
print("Tool-comparison scaffold — ProteinMPNN vs ESM-IF (vs FAMPNN if available). "
      "Verify the current public releases at generation time.")

## D3 (part 2) checklist
- [ ] Per-setting table: recovery, recapitulation rate, entropy, solubility proxies (`results/settings_summary.csv`).
- [ ] Pareto-optimal settings identified + the trade-off figure (foldability ↔ diversity ↔ solubility).
- [ ] Consensus-vs-single win rate reported honestly (incl. losses).
- [ ] ProteinMPNN vs ESM-IF comparison (per-tool distributions, N per tool).
- [ ] Honest per-setting hit rate (N recapitulating / N generated). No single "best" claimed.

**Next:** `05_validation_plan.ipynb` — the cheat-sheet + the wet-lab plan.